In [3]:
import polars as pl
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import json
import os
from pathlib import Path


def _repo_root() -> Path:
    env = os.environ.get("DICTYCITE_ROOT", "").strip()
    if env:
        return Path(env).expanduser().resolve()

    starts: list[Path] = []
    try:
        starts.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    starts.append(Path.cwd().resolve())

    for start in starts:
        p = start
        while True:
            if (p / ".git").exists():
                return p
            if p.parent == p:
                break
            p = p.parent

    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent, *cwd.parents):
        marker = base / "output" / "dicty_gold_build" / "1_curator_claims.parquet"
        if marker.is_file():
            return base
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd


def _output_gold_build() -> Path:
    return _repo_root() / "output" / "dicty_gold_build"


OUTPUT_GOLD_BUILD = _output_gold_build()

# citation-claims

In [4]:
claims = pl.read_parquet(OUTPUT_GOLD_BUILD / "1_curator_claims.parquet")
claims.head()

gene_id,sentence_markers,sentence_plain,cited_sentence_marked,claim_plain,anchors,publication_ids,citation_captions,citation_years
str,str,str,str,str,list[struct[2]],list[i64],list[str],list[i64]
"""DDB_G0287681""","""grlR encodes a member of the e…","""grlR encodes a member of the e…","""grlR encodes a member of the e…","""grlR encodes a member of the e…","[{126,[827]}]",[827],"[""Prabu and Eichinger 2006)""]",[2006]
"""DDB_G0290825""","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","[{144,[956]}, {253,[2919]}]","[956, 2919]","[""(Austin et al. 2006)"", ""(Thompson and Kay, 2000)""]","[2006, 2000]"
"""DDB_G0285319""","""There are three histone H3prot…","""There are three histone H3prot…","""There are three histone H3prot…","""There are three histone H3prot…","[{39,[4032]}]",[4032],"[""Bukenberger et al. 1992""]",[1992]
"""DDB_G0292810""","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","[{112,[9912]}]",[9912],"[""Dimond and Loomis 1976)""]",[1976]
"""DDB_G0292206""","""Expression is dependent on a s…","""Expression is dependent on a s…","""Expression is dependent on a s…","""Expression is dependent on a s…","[{126,[6149]}]",[6149],"[""Schatzle et al. 1991)""]",[1991]


In [5]:
# look at a row closely
row = claims.row(1, named=True)
print(row)

{'gene_id': 'DDB_G0290825', 'sentence_markers': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) [[PUB:956]] and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH [[PUB:2919]].', 'sentence_plain': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) (Austin et al. 2006) and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH (Thompson and Kay, 2000).', 'cited_sentence_marked': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) [CITE:956] and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH [CITE:2919].', 'claim_plain': 'The polyketide synthase S

In [6]:
# get preferred column set (same as before)
df = claims.select([
    "claim_plain",
    "anchors",
    "publication_ids",
    "citation_captions",
    "gene_id",
])

In [7]:
# clean up parentheses in citation_captions
df = df.with_columns(
    pl.col("citation_captions")
    .list.eval(
        pl.element()
        .str.replace_all(r"[()]", "")   # remove parentheses
        .str.replace_all(r"\s+", " ")   # collapse whitespace
        .str.strip_chars()              # <-- instead of .str.strip()
    )
    .alias("citation_captions")
)
# clean up comma
df = df.with_columns(
    pl.col("citation_captions")
    .list.eval(
        pl.element()
        .str.replace_all(r"[()]", "")
        .str.replace_all(r"\bet al\b\.?", "et al.")
        .str.replace_all(r",\s*et al\.", " et al.")
        .str.replace_all(r"et al\.\s*,\s*", "et al. ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )
    .alias("citation_captions")
)

In [8]:
# clean up very short claims, it shall have at least 6
min_words = 6

df2 = df.with_columns(
    # Count "words" by matching non-space sequences
    pl.col("claim_plain")
      .str.count_matches(r"\S+")
      .alias("n_words")
)

# 1) summary
summary = df2.select([
    pl.len().alias("rows_total"),
    (pl.col("n_words") < min_words).sum().alias("rows_to_drop"),
    (pl.col("n_words") >= min_words).sum().alias("rows_to_keep"),
])
print(summary)
# 2) inspect what you'd drop (optional)
to_drop = (
    df2.filter(pl.col("n_words") < min_words)
       .select(["n_words", "claim_plain", "gene_id"])
       .sort(["n_words", "claim_plain"])
)
with pl.Config(fmt_str_lengths=300):
    display(to_drop.head(50))

# 3) actually filter
df_filtered = df2.filter(pl.col("n_words") >= min_words).drop("n_words")

shape: (1, 3)
┌────────────┬──────────────┬──────────────┐
│ rows_total ┆ rows_to_drop ┆ rows_to_keep │
│ ---        ┆ ---          ┆ ---          │
│ u32        ┆ u32          ┆ u32          │
╞════════════╪══════════════╪══════════════╡
│ 2677       ┆ 24           ┆ 2653         │
└────────────┴──────────────┴──────────────┘


n_words,claim_plain,gene_id
u32,str,str
1,""".""","""DDB_G0284845"""
1,"""pneumoniae.""","""DDB_G0267444"""
1,"""pneumoniae.""","""DDB_G0267630"""
1,"""pneumoniae.""","""DDB_G0279183"""
2,"""(gskA) (..""","""DDB_G0281385"""
…,…,…
5,"""STATa is downregulated by PTP1.""","""DDB_G0281381"""
5,"""Spores have lower cellulose levels.""","""DDB_G0281387"""
5,"""at the tipped aggregate stage.""","""DDB_G0286185"""


In [9]:
# let us examine the duplicated claims closely
# 1) merge duplicates by concatenating gene_id (unique + sorted)
merged = (
    df_filtered
    .group_by([
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions"
    ])
    .agg(
        pl.col("gene_id")
          .unique()
          .sort()
          .str.join(",")
          .alias("gene_id")
    )
    .select([
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions",
        "gene_id",
    ])
)

print("after merge:")
print("rows =", merged.height)
print("unique claim_plain =", merged.select(pl.col("claim_plain").n_unique()).item())

# 2) check if claim_plain is STILL duplicated (i.e., same claim text but different citation/anchors/etc.)
still_dups = (
    merged
    .with_columns(pl.len().over("claim_plain").alias("n"))
    .filter(pl.col("n") > 1)
    .sort(["claim_plain"])
)

print("still duplicated claim_plain rows =", still_dups.height)

# show them grouped together if any remain
still_dups

after merge:
rows = 2107
unique claim_plain = 2103
still duplicated claim_plain rows = 8


claim_plain,anchors,publication_ids,citation_captions,gene_id,n
str,list[struct[2]],list[i64],list[str],str,u32
"""In Dictyostelium discoideum an…","[{183,[5046]}]",[5046],"[""Hauser et al. 1995""]","""DDB_G0267402,DDB_G0270838,DDB_…",2
"""In Dictyostelium discoideum an…","[{183,[5046]}]",[5046],"[""Hauser and colleagues 1995""]","""DDB_G0285319""",2
"""The 2C gene as well as 5 relat…","[{149,[10129]}]",[10129],"[""Schilde et al. 2004""]","""DDB_G0280871,DDB_G0280953,DDB_…",2
"""The 2C gene as well as 5 relat…","[{149,[1381]}]",[1381],"[""Schilde et al. 2004""]","""DDB_G0280847""",2
"""These mutant cells are also im…","[{148,[1548]}]",[1548],"[""Zhang et al. 2003""]","""DDB_G0279413""",2
"""These mutant cells are also im…","[{148,[2842]}]",[2842],"[""Wessels et al. 2000""]","""DDB_G0284331""",2
"""abcF1 and abcF4 are the most c…","[{115,[8189]}]",[8189],"[""Anjard and Loomis 2002""]","""DDB_G0267436,DDB_G0275637,DDB_…",2
"""abcF1 and abcF4 are the most c…","[{115,[2279]}]",[2279],"[""Anjard and Loomis 2002""]","""DDB_G0285997""",2


In [10]:
# with pl.Config(fmt_str_lengths=60):
#     display(still_dups)       

There are some duplicated claims.

Except the first one, the others are claims with inconsistant citations, let us remove those except the first one.

In [11]:
# 1) add a stable row id to merged
merged_i = merged.with_row_count("row_nr")   # if this errors on your Polars, use: .with_row_index("row_nr")

# 2) recompute still_dups from merged_i (so it contains row_nr)
still_dups_i = (
    merged_i
    .with_columns(pl.len().over("claim_plain").alias("n"))
    .filter(pl.col("n") > 1)
    .sort(["claim_plain", "row_nr"])
)

# 3) drop everything in still_dups except the first row of still_dups
drop_row_nrs = still_dups_i.slice(1).select("row_nr")   # everything after the first row

merged_clean = (
    merged_i
    .join(drop_row_nrs, on="row_nr", how="anti")  # remove those rows
    .drop(["row_nr", "n"], strict=False)
)

/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_22397/3282369987.py:2: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  merged_i = merged.with_row_count("row_nr")   # if this errors on your Polars, use: .with_row_index("row_nr")


In [12]:
merged_clean

claim_plain,anchors,publication_ids,citation_captions,gene_id
str,list[struct[2]],list[i64],list[str],str
"""pkaC encodes the catalytic sub…","[{77,[5939]}, {78,[5469]}]","[5939, 5469]","[""Mann et al. 1992"", ""Anjard et al. 1993""]","""DDB_G0283907"""
"""These results suggest an inter…","[{145,[11775]}]",[11775],"[""Gopaldass et al. 2011""]","""DDB_G0277849"""
"""In addition, sodC null cells h…","[{374,[587]}]",[587],"[""Veeranki et al. 2008""]","""DDB_G0282993"""
"""Additionally, yeast two-hybrid…","[{127,[1640]}]",[1640],"[""Aubry et al. 2003""]","""DDB_G0289555"""
"""arpB encodes the 44 kDa actin-…","[{67,[2417]}]",[2417],"[""Insall et al. 2001""]","""DDB_G0272106"""
…,…,…,…,…
"""Furthermore NMR and mass spec …","[{178,[16786, 17089, 18528]}]","[16786, 17089, 18528]","[""Sheikh et al. 2017"", ""Xu et al. 2018"", ""West and Kim 2019""]","""DDB_G0269230,DDB_G0273251,DDB_…"
"""In addition, gapA null cells h…","[{235,[12093]}]",[12093],"[""Kee et al. 2012""]","""DDB_G0269140"""
"""An fhbA / fhbB double null mut…","[{235,[3038]}]",[3038],"[""Iijima et al. 2000""]","""DDB_G0292378"""


In [13]:
# give claim ID, and we want to explode the list columns together to get one row per (claim_id, publication_id)
m = merged_clean.with_columns(
    pl.col("claim_plain").rank(method="dense").cast(pl.Int64).alias("claim_id")
).select([
    "claim_id",
    "claim_plain",
    "anchors",
    "publication_ids",
    "citation_captions",
    "gene_id"
])

In [14]:
# 2) sanity check: list columns must be aligned per row before explode (len(citation_captions) == len(publication_ids))
bad_rows = (
    m
    .with_columns([
        pl.col("publication_ids").list.len().alias("n_pub"),
        pl.col("citation_captions").list.len().alias("n_cap")
    ])
    .filter(
        (pl.col("n_pub") != pl.col("n_cap")) 
    )
    .select([
        "claim_id",
        "gene_id",
        "n_pub", "n_cap", 
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions"
    ])
    .sort(["n_pub", "n_cap"], descending=True)
)

print("Number of misaligned rows:", bad_rows.height)

with pl.Config(fmt_str_lengths=1000):
    display(bad_rows)

Number of misaligned rows: 4


claim_id,gene_id,n_pub,n_cap,claim_plain,anchors,publication_ids,citation_captions
i64,str,u32,u32,str,list[struct[2]],list[i64],list[str]
1195,"""DDB_G0286183""",2,1,"""The Dictyostelium Agps protein has been crystallized.""","[{52,[712, 659]}]","[712, 659]","[""Razeto et al. ., 2007""]"
1534,"""DDB_G0287031""",2,1,"""These results reveal that gpaC functions as a regulator of early development gene expression,.""","[{92,[4035]}, {93,[4034]}]","[4035, 4034]","[""Brandon et al. 1997""]"
1469,"""DDB_G0293084""",2,1,"""The zizB mutant phenotypes and ZizB binding partners suggest a central role for ZizB in actin cytoskeletal organization and cortical stabilization.,.""","[{147,[12098]}, {148,[12830]}]","[12098, 12830]","[""Pakes et al. 2012""]"
471,"""DDB_G0268620""",2,1,"""Fluid uptake by macropinocytosis is reduced about 3 fold and phagosomes remain more acidic in pkbA - null cells (, (.""","[{113,[2428]}, {116,[2634]}]","[2428, 2634]","[""Rupper et al. 2001""]"


in the 4 cases here, len(publication_ids)>len(citation_captions), after closer look, it is the same author and same year, we manully make a patch here

In [15]:
m_fixed = (
    m.with_columns(
        pl.when(pl.col("claim_id") == 1534)
          .then(pl.lit(["Brandon et al. 1997a", "Brandon et al. 1997b"]))
        .when(pl.col("claim_id") == 471)
          .then(pl.lit(["Rupper et al. 2001a", "Rupper et al. 2001b"]))
        .when(pl.col("claim_id") == 1469)
          .then(pl.lit(["Pakes et al. 2012a", "Pakes et al. 2012b"]))
        .when(pl.col("claim_id") == 1195)
          .then(pl.lit(["Razeto et al. 2007a", "Razeto et al. 2007b"]))
        .otherwise(pl.col("citation_captions"))
        .alias("citation_captions")
    )
)

In [16]:
m_long = (
    m_fixed
    .explode(["publication_ids", "citation_captions"])
    .rename({"publication_ids": "publication_id"})
)

In [17]:
m_long

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id
i64,str,list[struct[2]],i64,str,str
1985,"""pkaC encodes the catalytic sub…","[{77,[5939]}, {78,[5469]}]",5939,"""Mann et al. 1992""","""DDB_G0283907"""
1985,"""pkaC encodes the catalytic sub…","[{77,[5939]}, {78,[5469]}]",5469,"""Anjard et al. 1993""","""DDB_G0283907"""
1539,"""These results suggest an inter…","[{145,[11775]}]",11775,"""Gopaldass et al. 2011""","""DDB_G0277849"""
694,"""In addition, sodC null cells h…","[{374,[587]}]",587,"""Veeranki et al. 2008""","""DDB_G0282993"""
101,"""Additionally, yeast two-hybrid…","[{127,[1640]}]",1640,"""Aubry et al. 2003""","""DDB_G0289555"""
…,…,…,…,…,…
498,"""Furthermore NMR and mass spec …","[{178,[16786, 17089, 18528]}]",18528,"""West and Kim 2019""","""DDB_G0269230,DDB_G0273251,DDB_…"
671,"""In addition, gapA null cells h…","[{235,[12093]}]",12093,"""Kee et al. 2012""","""DDB_G0269140"""
140,"""An fhbA / fhbB double null mut…","[{235,[3038]}]",3038,"""Iijima et al. 2000""","""DDB_G0292378"""


In [18]:
# now we exlpode anchors
# 1) Build mapping: (claim_id, publication_id) -> anchor_pos
anchor_map = (
    m_fixed
    .select(["claim_id", "anchors"])
    .explode("anchors")  # now each row is one struct {pos, pub_ids}
    .with_columns([
        pl.col("anchors").struct.field("pos").alias("anchor_pos"),
        pl.col("anchors").struct.field("pub_ids").alias("publication_id"),
    ])
    .explode("publication_id")  # one row per pub_id
    .select(["claim_id", "publication_id", "anchor_pos"])
)

# If there can be multiple anchor_pos for the same (claim_id, publication_id), keep them all:
anchor_map = (
    anchor_map
    .group_by(["claim_id", "publication_id"])
    .agg(pl.col("anchor_pos").sort().alias("anchor_pos"))
)

# 2) Join onto m_long
m_long2 = m_long.join(anchor_map, on=["claim_id", "publication_id"], how="left")

# 3) If you want exactly one row per (claim_id, publication_id, anchor_pos), explode anchor_pos:
m_long3 = m_long2.explode("anchor_pos")

# Final columns (example)
m_long3.select([
    "claim_id",
    "claim_plain",
    "anchor_pos",
    "publication_id",
    "citation_captions",
    "gene_id",
])

claim_id,claim_plain,anchor_pos,publication_id,citation_captions,gene_id
i64,str,i64,i64,str,str
1985,"""pkaC encodes the catalytic sub…",77,5939,"""Mann et al. 1992""","""DDB_G0283907"""
1985,"""pkaC encodes the catalytic sub…",78,5469,"""Anjard et al. 1993""","""DDB_G0283907"""
1539,"""These results suggest an inter…",145,11775,"""Gopaldass et al. 2011""","""DDB_G0277849"""
694,"""In addition, sodC null cells h…",374,587,"""Veeranki et al. 2008""","""DDB_G0282993"""
101,"""Additionally, yeast two-hybrid…",127,1640,"""Aubry et al. 2003""","""DDB_G0289555"""
…,…,…,…,…,…
498,"""Furthermore NMR and mass spec …",178,18528,"""West and Kim 2019""","""DDB_G0269230,DDB_G0273251,DDB_…"
671,"""In addition, gapA null cells h…",235,12093,"""Kee et al. 2012""","""DDB_G0269140"""
140,"""An fhbA / fhbB double null mut…",235,3038,"""Iijima et al. 2000""","""DDB_G0292378"""


In [19]:
# now we get year from citation_captions
YEAR_RE = r"(18|19|20)\d{2}"

m_long3 = m_long3.with_columns(
    year=pl.col("citation_captions")
        .str.extract(YEAR_RE, 0)   # whole match
        .cast(pl.Int32)
)
m_long3

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year
i64,str,list[struct[2]],i64,str,str,i64,i32
1985,"""pkaC encodes the catalytic sub…","[{77,[5939]}, {78,[5469]}]",5939,"""Mann et al. 1992""","""DDB_G0283907""",77,1992
1985,"""pkaC encodes the catalytic sub…","[{77,[5939]}, {78,[5469]}]",5469,"""Anjard et al. 1993""","""DDB_G0283907""",78,1993
1539,"""These results suggest an inter…","[{145,[11775]}]",11775,"""Gopaldass et al. 2011""","""DDB_G0277849""",145,2011
694,"""In addition, sodC null cells h…","[{374,[587]}]",587,"""Veeranki et al. 2008""","""DDB_G0282993""",374,2008
101,"""Additionally, yeast two-hybrid…","[{127,[1640]}]",1640,"""Aubry et al. 2003""","""DDB_G0289555""",127,2003
…,…,…,…,…,…,…,…
498,"""Furthermore NMR and mass spec …","[{178,[16786, 17089, 18528]}]",18528,"""West and Kim 2019""","""DDB_G0269230,DDB_G0273251,DDB_…",178,2019
671,"""In addition, gapA null cells h…","[{235,[12093]}]",12093,"""Kee et al. 2012""","""DDB_G0269140""",235,2012
140,"""An fhbA / fhbB double null mut…","[{235,[3038]}]",3038,"""Iijima et al. 2000""","""DDB_G0292378""",235,2000


In [20]:
m_long4 = m_long3.select([
    "claim_id",
    "claim_plain",
    "anchors",        # keep if you still want the original struct list
    "anchor_pos",
    "publication_id",
    "citation_captions",
    "gene_id",
    "year",
])

i still see some claims needs cleaned up. They are almost identical with only dot diffrenece.


In [21]:
claim_key_expr = (
    pl.col("claim_plain")
    .str.to_lowercase()
    .str.replace_all(r"\.{2,}", ".")          # ".." / "..." -> "."
    .str.replace_all(r"[^a-z0-9\s]", " ")     # drop punctuation
    .str.replace_all(r"\s+", " ")             # collapse whitespace
    .str.strip_chars()
)

tmp = m_long4.with_columns(
    claim_key=claim_key_expr
)

# Optional: see groups where multiple raw claim_plain map to same key
near_same = (
    tmp.group_by("claim_key")
       .agg([
           pl.len().alias("n_rows"),
           pl.col("claim_plain").n_unique().alias("n_variants"),
           pl.col("claim_plain"),
       ])
       .filter(pl.col("n_variants") > 1)
       .sort("n_rows", descending=True)
)
near_same

claim_key,n_rows,n_variants,claim_plain
str,u32,u32,list[str]
"""at the trailing edge activated…",9,2,"[""At the trailing edge activated Rac1 forms a complex with DGAP1 and the actin-binding proteins cortexillin I (ctxA) and cortexillin II (ctxB) enabling proper cell retraction,,,,."", ""At the trailing edge activated Rac1 forms a complex with DGAP1 and the actin-binding proteins cortexillin I (ctxA) and cortexillin II (ctxB) enabling proper cell retraction,,,,."", … ""At the trailing edge activated Rac1 forms a complex with DGAP1 and the actin-binding proteins cortexillin I (ctxA) and cortexillin II (ctxB) enabling proper cell retraction,,,..""]"
"""the major spore coat proteins …",9,2,"[""The major spore coat proteins, SP96, SP70, and SP60 ( cotA, cotB, cotC, respectively), are coordinately synthesized in prespore cells and stored in prespore vesicles (PSVs);;;,."", ""The major spore coat proteins, SP96, SP70, and SP60 ( cotA, cotB, cotC, respectively), are coordinately synthesized in prespore cells and stored in prespore vesicles (PSVs);;;,."", … ""The major spore coat proteins, SP96, SP70, and SP60 ( cotA, cotB, cotC, respectively), are coordinately synthesized in prespore cells and stored in prespore vesicles (PSVs);;,.""]"
"""expression of cota sp96 cotb s…",6,2,"[""Expression of cotA (SP96), cotB (SP70) and cotC (SP60) has been used to monitor prespore specific transcription in a large number of studies (e.g.,;;.."", ""Expression of cotA (SP96), cotB (SP70) and cotC (SP60) has been used to monitor prespore specific transcription in a large number of studies (e.g.,;;.."", … ""Expression of cotA (SP96), cotB (SP70) and cotC (SP60) has been used to monitor prespore specific transcription in a large number of studies (e.g.,;;.""]"
"""dictyostelium has three genes …",6,3,"[""Dictyostelium has three genes encoding profilin, proA, (proB) and (proC),."", ""Dictyostelium has three genes encoding profilin, proA, (proB) and (proC),."", … ""Dictyostelium has three genes encoding profilin, (proA), (proB) and proC,.""]"
"""about 100 different proteins c…",6,6,"[""About 100 different proteins contribute to centrosomal functions including the products of (tubC), (spc97), (spc98), (cenA), (lis1), (nek2), (eb1), (cepJ) and cepG..."", ""About 100 different proteins contribute to centrosomal functions including the products of (tubC), (spc97), (spc98), (cenA), lis1, (nek2), (eb1), (cepJ) and (cepG)."", … ""About 100 different proteins contribute to centrosomal functions including the products of tubC, spc97, (spc98), (cenA), (lis1), (nek2), (eb1), (cepJ) and (cepG)..""]"
…,…,…,…
"""dictyostelium and arabidopsis …",2,2,"[""Dictyostelium and Arabidopsis have members of both types."", ""Dictyostelium and Arabidopsis have members of both types..""]"
"""these results suggest the exis…",2,2,"[""These results suggest the existence of an unidentified sulfated factor playing a key role in the intracellular killing of Klebsiella bacteria."", ""These results suggest the existence of an unidentified sulfated factor playing a key role in the intracellular killing of Klebsiella bacteria..""]"
"""tandem duplications on chromos…",2,2,"[""Tandem duplications on chromosome 4 generated ( tagB ), tagC, and ( tagD ) which all encode membrane-embedded proteases (."", ""Tandem duplications on chromosome 4 generated tagB, ( tagC ), and ( tagD ) which all encode membrane-embedded proteases (.""]"


yes, there are actually quite many near duplicates. shall be also merged

In [22]:
# 2) Create a new merged claim_id based on claim_key
tmp = tmp.with_columns(
    claim_id_new=pl.col("claim_key").rank(method="dense").cast(pl.Int64)
)

# 3) Merge rows safely and keep your column order
#    - choose a canonical claim_plain (first seen)
#    - merge gene_id as union joined by ","
claim_cleaned = (
    tmp
    .group_by([
        "claim_id_new",
        "publication_id",
        "citation_captions",
        "anchor_pos",
        "year",
    ])
    .agg([
        pl.first("claim_plain").alias("claim_plain"),
        pl.first("anchors").alias("anchors"),
        pl.col("gene_id").unique().sort().str.join(",").alias("gene_id"),
    ])
    .select([
        pl.col("claim_id_new").alias("claim_id"),
        "claim_plain",
        "anchors",
        "publication_id",
        "citation_captions",
        "gene_id",
        "anchor_pos",
        "year",
    ])
    .sort(["claim_id", "publication_id", "anchor_pos"])
)

claim_cleaned

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year
i64,str,list[struct[2]],i64,str,str,i64,i32
1,"""A basic region in the tail is …","[{94,[13954]}]",13954,"""Brzeska et al. 2014""","""DDB_G0289117""",94,2014
2,"""A cDNA clone derived from psvA…","[{102,[8316]}]",8316,"""Barklis and Lodish 1983""","""DDB_G0276869""",102,1983
3,"""A cDNA clone of this gene, D19…","[{89,[8283]}]",8283,"""Chisholm et al. 1984""","""DDB_G0267412""",89,1984
4,"""A co-immunoprecipitation assay…","[{158,[19729]}]",19729,"""Li et al. 2020""","""DDB_G0274607""",158,2020
5,"""A comparison of the genes tran…","[{185,[13151]}]",13151,"""Galardi-Castilla et al. 2013""","""DDB_G0268920""",185,2013
…,…,…,…,…,…,…,…
2059,"""Yet another role for NDP kinas…","[{131,[4899]}]",4899,"""Sonnemann and Mutzel 1995""","""DDB_G0273069,DDB_G0273805""",131,1995
2060,"""zizA null mutant cells do not …","[{72,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0275035""",72,2012
2061,"""ZizB also interacts with sever…","[{106,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0267374,DDB_G0272104""",106,2012


In [23]:
year_summary = (
    claim_cleaned
    .with_columns(pl.col("year").fill_null(-1).alias("year2"))
    .group_by("year2")
    .agg(pl.len().alias("n"))
    .sort("year2")
    .with_columns(
        pl.when(pl.col("year2") == -1).then(None).otherwise(pl.col("year2")).alias("year")
    )
    .select(["year", "n"])
)

year_summary

year,n
i32,u32
null,7
1956,1
1965,1
1967,3
1968,1
…,…
2016,76
2017,58
2018,81


In [26]:
# claim_cleaned.write_parquet(...)  # intermediate; dropped from output

In [27]:
claim_cleaned.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)

n_unique_claim_id
u32
2063


In [28]:
# claim_cleaned TSV export removed (intermediate; dropped from output)

I still see some duplicated claims. There are still a few edge cases. But at this point, I am not gonna trying to “perfectly” clean. I will try to pick up a golden set out of it.

In [29]:
# manual inspection, will pick some golden set maunally later
claim_cleaned_manual = claim_cleaned.filter(pl.col("anchor_pos") != 1)
# claim_cleaned_gold=...

# match pmid

In [30]:
pub_pmid = pl.read_csv(OUTPUT_GOLD_BUILD / "2_publication_id_pmid.csv")
pub_pmid

publication_id,pmid
i64,i64
12,21243421
13,21239624
15,21235525
17,20950684
20,21150268
…,…
19689,32769116
19708,21551065
19728,32821814


In [31]:
# check if all publcation id can be mapped to pmid

# pick the right df names
a = claim_cleaned
b = pub_pmid

# make sure both publication_id columns are the same type (I recommend Int64)
a_ids = a.select(pl.col("publication_id").cast(pl.Int64)).unique()
b_ids = b.select(pl.col("publication_id").cast(pl.Int64)).unique()

# overlap + only-in sets
overlap = a_ids.join(b_ids, on="publication_id", how="inner")
only_a  = a_ids.join(b_ids, on="publication_id", how="anti")
only_b  = b_ids.join(a_ids, on="publication_id", how="anti")

summary = pl.DataFrame({
    "set": ["claim_cleaned", "pub_pmid", "overlap", "only_claim_cleaned", "only_pub_pmid"],
    "unique_count": [a_ids.height, b_ids.height, overlap.height, only_a.height, only_b.height],
})

summary

set,unique_count
str,i64
"""claim_cleaned""",1400
"""pub_pmid""",4341
"""overlap""",1373
"""only_claim_cleaned""",27
"""only_pub_pmid""",2968


27 ids are not mapped.

In [32]:
claim_cleaned_pmid = (
    claim_cleaned
    .with_columns(pl.col("publication_id").cast(pl.Int64))
    .join(pub_pmid, on="publication_id", how="left")
    .with_columns(
        pl.col("pmid").fill_null("NA")  # or keep as null if you prefer
    )
)

claim_cleaned_pmid

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year,pmid
i64,str,list[struct[2]],i64,str,str,i64,i32,str
1,"""A basic region in the tail is …","[{94,[13954]}]",13954,"""Brzeska et al. 2014""","""DDB_G0289117""",94,2014,"""24747353"""
2,"""A cDNA clone derived from psvA…","[{102,[8316]}]",8316,"""Barklis and Lodish 1983""","""DDB_G0276869""",102,1983,"""6301681"""
3,"""A cDNA clone of this gene, D19…","[{89,[8283]}]",8283,"""Chisholm et al. 1984""","""DDB_G0267412""",89,1984,"""6429548"""
4,"""A co-immunoprecipitation assay…","[{158,[19729]}]",19729,"""Li et al. 2020""","""DDB_G0274607""",158,2020,"""32818671"""
5,"""A comparison of the genes tran…","[{185,[13151]}]",13151,"""Galardi-Castilla et al. 2013""","""DDB_G0268920""",185,2013,"""23577638"""
…,…,…,…,…,…,…,…,…
2059,"""Yet another role for NDP kinas…","[{131,[4899]}]",4899,"""Sonnemann and Mutzel 1995""","""DDB_G0273069,DDB_G0273805""",131,1995,"""7733916"""
2060,"""zizA null mutant cells do not …","[{72,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0275035""",72,2012,"""22366457"""
2061,"""ZizB also interacts with sever…","[{106,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0267374,DDB_G0272104""",106,2012,"""22366457"""


In [33]:
# claim_cleaned_pmid.write_parquet(...)  # intermediate; dropped from output

claim_cleaned_pmid_nonNA = claim_cleaned_pmid.filter(
    pl.col("pmid").is_not_null() & (pl.col("pmid") != "NA")
)

# claim_cleaned_pmid_nonNA.write_parquet(...)  # intermediate; dropped from output

In [34]:
claim_cleaned_pmid_nonNA.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)

n_unique_claim_id
u32
2040


In [35]:
claim_cleaned_pmid_nonNA.select(
    pl.col("pmid").n_unique().alias("n_unique_pmid")
)

n_unique_pmid
u32
1372


# how many we claims having abstracts on EPMC

In [37]:
EPMC = pl.read_parquet(OUTPUT_GOLD_BUILD / "3_articles_cleaned_abstract.parquet")

In [38]:
EPMC

pmid,pmcid,doi,year,title,journal,authors,abstract_clean,file
str,str,str,str,str,str,str,str,str
"""2654141""","""PMC2115546""","""10.1083/jcb.108.5.1751""","""1989""","""Centrin-mediated microtubule s…","""The Journal of cell biology""","""Sanders MA, Salisbury JL.""","""Chlamydomonas cells excise the…","""article_fetching/output/all_cl…"
"""39528565""","""PMC11555045""","""10.1038/s41467-024-54272-4""","""2024""","""Nuclear localization sequence …","""Nature communications""","""Lim YJ, Yoon YJ, Lee H, Choi G…","""Plant pathogens secrete nuclea…","""article_fetching/output/all_cl…"
"""6319129""",null,"""10.1111/j.1432-1033.1984.tb078…","""1984""","""Investigations on stimulation …","""European journal of biochemist…","""Scholübbers HG, van Knippenber…","""The ability of 24 systematical…","""article_fetching/output/all_cl…"
"""28740830""","""PMC5502327""","""10.3389/fonc.2017.00139""","""2017""","""Structure, Activity Regulation…","""Frontiers in oncology""","""Mammucari C, Gherardi G, Rizzu…","""Mitochondrial Ca2+ uptake play…","""article_fetching/output/all_cl…"
"""18814278""",null,"""10.1002/cm.20314""","""2008""","""Correlated waves of actin fila…","""Cell motility and the cytoskel…","""Asano Y, Nagasaki A, Uyeda TQ.""","""Chemotaxis-deficient amiB-null…","""article_fetching/output/all_cl…"
…,…,…,…,…,…,…,…,…
"""36543032""","""PMC9889102""","""10.1016/j.ejmech.2022.115008""","""2023""","""Isoform selectivities of novel…","""European journal of medicinal …","""Smith JD, Brawley J, Bordenave…","""Muscle myosin inhibition could…","""article_fetching/output/all_cl…"
"""20023070""","""PMC2823002""","""10.1128/ec.00220-09""","""2010""","""Distinct subcellular localizat…","""Eukaryotic cell""","""Schilde C, Schönemann B, Sehri…","""We have identified new synapto…","""article_fetching/output/all_cl…"
"""10706824""",null,"""10.1006/scdb.1999.0343""","""1999""","""Control of spatial patterning …","""Seminars in cell & development…","""Mohanty S, Firtel RA.""","""The spatial patterning of pres…","""article_fetching/output/all_cl…"


In [39]:
a = claim_cleaned_pmid_nonNA.select(pl.col("pmid").cast(pl.Utf8).alias("pmid")).unique()
b = EPMC.select(pl.col("pmid").cast(pl.Utf8).alias("pmid")).unique()

overlap = a.join(b, on="pmid", how="inner")
only_a  = a.join(b, on="pmid", how="anti")
only_b  = b.join(a, on="pmid", how="anti")

summary = pl.DataFrame({
    "set": ["claim_cleaned_pmid_nonNA", "epmc_df", "overlap", "only_claim_cleaned_pmid", "only_epmc_df"],
    "unique_count": [a.height, b.height, overlap.height, only_a.height, only_b.height],
})

summary

set,unique_count
str,i64
"""claim_cleaned_pmid_nonNA""",1372
"""epmc_df""",20447
"""overlap""",1340
"""only_claim_cleaned_pmid""",32
"""only_epmc_df""",19107


In [40]:
only_claim_pmids = only_a.get_column("pmid")

rows_only_claim = claim_cleaned_pmid.filter(
    pl.col("pmid").cast(pl.Utf8).is_in(only_claim_pmids)
)

rows_only_claim

/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_22397/496506703.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  rows_only_claim = claim_cleaned_pmid.filter(


claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year,pmid
i64,str,list[struct[2]],i64,str,str,i64,i32,str
98,"""Adenylyl cyclase increases mor…","[{133,[9781]}, {136,[10027]}, {139,[9617]}]",9781,"""Roos et al. 1977""","""DDB_G0281545""",133,1977,"""191758"""
98,"""Adenylyl cyclase increases mor…","[{133,[9781]}, {136,[10027]}, {139,[9617]}]",10027,"""Klein 1976""","""DDB_G0281545""",136,1976,"""183985"""
200,"""axeB was identified as a mutan…","[{139,[10264]}, {142,[7409]}]",10264,"""Williams et al. 1974""","""DDB_G0350652""",139,1974,"""4474353"""
206,"""Biochemical assays found a 10 …","[{114,[10751]}]",10751,"""Ashworth and Sussman, 1967""","""DDB_G0289875""",114,1967,"""6067273"""
207,"""Biochemical assays found a 10 …","[{126,[10751]}]",10751,"""Ashworth and Sussman, 1967""","""DDB_G0277879""",126,1967,"""6067273"""
…,…,…,…,…,…,…,…,…
1661,"""The major spore coat proteins,…","[{172,[9387]}, {173,[8541]}, … {176,[6031]}]",6653,"""Fosnaugh and Loomis, 1989""","""DDB_G0276941""",175,1989,"""2587278"""
1661,"""The major spore coat proteins,…","[{172,[9387]}, {173,[8541]}, … {176,[6031]}]",9387,"""Orlowski and Loomis, 1979""","""DDB_G0276761,DDB_G0277141,DDB_…",172,1979,"""499661"""
1783,"""The specific activity of glyco…","[{201,[10721]}, {202,[9996]}]",10721,"""Wright and Dalhberg, 1967""","""DDB_G0267674""",201,1967,"""6069170"""


Some claims are not covered. It seems that most of them are old paper that there are no abstract available. So I remove those and merge title and abstract to the claim table

In [41]:
epmc_small = EPMC.select([
    pl.col("pmid").cast(pl.Utf8).alias("pmid"),
    pl.col("title").alias("title"),
    pl.col("abstract_clean").alias("abstract_clean"),
])

claim_small = claim_cleaned_pmid_nonNA.with_columns(
    pl.col("pmid").cast(pl.Utf8).alias("pmid")
)

# inner join = ignore only-claim PMIDs
claim_cleaned_pmid_nonNA_abstract = claim_small.join(epmc_small, on="pmid", how="inner")

claim_cleaned_pmid_nonNA_abstract

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year,pmid,title,abstract_clean
i64,str,list[struct[2]],i64,str,str,i64,i32,str,str,str
1878,"""These results suggest that Atg…","[{175,[16288]}]",16288,"""Messling et al. 2017""","""DDB_G0286191,DDB_G0290491""",175,2017,"""28413119""","""The two Dictyostelium discoide…","""Autophagy is a highly conserve…"
403,"""Disruption of csaA results in …","[{103,[6630]}]",6630,"""Harloff et al. 1989""","""DDB_G0289073""",103,1989,"""2515990""","""Selective elimination of the c…","""The contact site A glycoprotei…"
1105,"""MyoB is non-filamentous and co…","[{163,[6608]}, {164,[6857]}, {165,[12097]}]",12097,"""Brzeska et al. 2012""","""DDB_G0289117""",165,2012,"""22367211""","""Molecular basis of dynamic rel…","""Class I myosins have a single …"
1196,"""pestis recruits host Rab1B pro…","[{235,[17146]}]",17146,"""Markman et al. 2018""","""DDB_G0277867""",235,2018,"""29350155""","""Yersinia pestis Survival and R…","""Plague ecology is characterize…"
582,"""FspA protein is an essential c…","[{122,[13550]}]",13550,"""Lima et al. 2013""","""DDB_G0277237""",122,2013,"""24128258""","""Two distinct sensing pathways …","""Recognition of bacteria by met…"
…,…,…,…,…,…,…,…,…,…,…
439,"""Dominant negative rabS cells d…","[{174,[17491]}]",17491,"""Yarbrough et al. 2018""","""DDB_G0282537""",174,2018,"""29843387""","""The Effect of Overexpressed Dd…","""Rab GTPases are essential regu…"
1309,"""Rab2A is associated with both …","[{83,[15422]}, {136,[17491]}]",17491,"""Yarbrough et al. 2018""","""DDB_G0282537""",136,2018,"""29843387""","""The Effect of Overexpressed Dd…","""Rab GTPases are essential regu…"
516,"""Expression of cotE is dependen…","[{169,[11602]}]",11602,"""Huang et al. 2011""","""DDB_G0277903""",169,2011,"""21810415""","""BzpF is a CREB-like transcript…","""The cAMP response element-bind…"


In [42]:
# claim_cleaned_pmid_nonNA_abstract.write_parquet(...)  # intermediate; dropped from output

In [43]:
claim_cleaned_pmid_nonNA_abstract.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)

n_unique_claim_id
u32
2020


In [44]:
claim_cleaned_pmid_nonNA_abstract.select(
    pl.col("pmid").n_unique().alias("n_unique_pmid")
)

n_unique_pmid
u32
1340


# clean up even more and help for narrow down to a golden set

In [45]:
claims_u = (
    claim_cleaned_pmid_nonNA_abstract.select(["claim_id", "claim_plain"])
        .unique(subset=["claim_id"])
        .sort("claim_id")
)
claims_u.height

2020

In [46]:
claims_u = claims_u.with_columns(
    claim_sim=(
        pl.col("claim_plain")
        # remove bracket chars
        .str.replace_all(r"[()\[\]{}]", "")
        # ; -> ,
        .str.replace_all(r";", ",")
        # collapse obvious runs
        .str.replace_all(r"\.{2,}", ".")
        .str.replace_all(r",(\s*,)+", ",")
        # fix mixed combos
        .str.replace_all(r",\s*\.", ".")
        .str.replace_all(r"\.\s*,", ".")
        # spacing rules
        .str.replace_all(r"\s+([,.])", r"$1")          # no space before , .
        .str.replace_all(r",([A-Za-z0-9])", r", $1")   # space after comma
        .str.replace_all(r"\.([A-Za-z])", r". $1")     # space after dot only before letters (avoids 3.14 -> 3. 14)
        # final whitespace cleanup
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )
)

In [47]:
# # take a look
# (
#     claims_u
#     .write_csv("tmp/claims_u.tsv", separator="\t")
# )

Cluster near-duplicates with TF-IDF char ngrams

In [48]:
texts = claims_u.get_column("claim_sim").to_list()
ids   = claims_u.get_column("claim_id").to_list()
n = len(ids)

# --- TF-IDF (char ngrams) ---
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
X = vec.fit_transform(texts)

# --- Nearest neighbors (candidate pairs) ---
k = 30  #  <-----------lower value might miss pairs, larger values might slow down
nn = NearestNeighbors(n_neighbors=min(k, n), metric="cosine").fit(X)
dist, idx = nn.kneighbors(X, return_distance=True)

# --- Build candidate pair list with similarities ---
# (we'll inspect these, and also use them for clustering)
pairs = []
for i in range(n):
    for d, j in zip(dist[i], idx[i]):
        if j <= i:
            continue
        sim = 1.0 - float(d)
        pairs.append((i, j, sim))

pairs.sort(key=lambda x: x[2], reverse=True)

# Inspect top similar pairs (e.g. top 100)
topN = 10                               # <-------------------change here for look longer list
pairs_df = pl.DataFrame(
    {
        "i": [p[0] for p in pairs[:topN]],
        "j": [p[1] for p in pairs[:topN]],
        "sim": [p[2] for p in pairs[:topN]],
        "claim_id_i": [ids[p[0]] for p in pairs[:topN]],
        "claim_id_j": [ids[p[1]] for p in pairs[:topN]],
        "text_i": [texts[p[0]] for p in pairs[:topN]],
        "text_j": [texts[p[1]] for p in pairs[:topN]],
    }
).sort("sim", descending=True)

with pl.Config(fmt_str_lengths=500, tbl_rows=topN, tbl_cols=20):
    display(pairs_df)

i,j,sim,claim_id_i,claim_id_j,text_i,text_j
i64,i64,f64,i64,i64,str,str
371,372,0.995934,381,382,"""Dictyostelium has 71 ABC transporters including one on the chromosome 2 duplication and 11 ABCA family members: abcA1, abcA2, abcA3, abcA4, abcA5, abcA6, abcA7, abcA8, abcA9, abcA10, and abcA11, One of the defining characteristics of the A family is the presence of a regulatory domain with multiple sites for phosphorylation by various protein kinases in the region after the first ABC domain.""","""Dictyostelium has 71 ABC transporters including one on the chromosome 2 duplication, including 11 ABCA family members: abcA1, abcA2, abcA3, abcA4, abcA5, abcA6, abcA7, abcA8, abcA9, abcA10, and abcA11, One of the defining characteristics of the A family is the presence of a regulatory domain with multiple sites for phosphorylation by various protein kinases in the region after the first ABC domain."""
1966,1967,0.992799,2008,2009,"""Triple mutant cells lacking myoD, myoE and myoF are defective in chemotaxis to cAMP, chemoattractant-stimulated actin polymerization and phagocytosis.""","""Triple mutants cells lacking myoD, myoE and myoF are defective in chemotaxis to cAMP, chemoattractant-stimulated actin polymerization and phagocytosis."""
1105,1186,0.992057,1132,1215,"""NapA and PirA interact to form an independent heterodimeric subcomplex prior to formation of the mature Scar/WAVE complex.""","""PirA and NapA also interact to form an independent heterodimeric subcomplex prior to formation of the mature Scar/WAVE complex."""
347,546,0.989565,357,563,"""demonstrate two distinct signaling pathways for pstA-specific genes downstream of STATa: CudA-dependent and CudA-independent.""","""For pstA-specific genes, demonstrate two distinct signaling pathways downstream of STATa: CudA-dependent and CudA-independent."""
474,475,0.984793,491,492,"""Endosomes are transported along microtubules with frequent reversals in direction due to a tug-of-war between dynein pulling to the minus-end and kinesin kif1 pulling to the plus-end.""","""Endosomes are transported along microtubules with frequent reversals in direction due to a tug-of-war between kinesin kif1 pulling to the plus-end and dynein dhcA pulling to the minus-end."""
236,1535,0.981453,245,1568,"""CARMIL localizes in dynamic actin-rich cellular extensions, including the leading edge of chemotaxing cells and macropinocytic extensions.""","""The CARMIL complex localizes in dynamic actin-rich cellular extensions, including the leading edge of chemotaxing cells and macropinocytic extensions."""
129,133,0.980474,135,139,"""An analysis of Dictyostelium ras family genes reveals that rasU rasU, rasV rasV, rasW rasW, rasX rasX, rasY rasY and rasZ form a separate phylogenetic clade.""","""An analysis of Dictyostelium ras family genes reveals that rasU, rasV rasV, rasW rasW, rasX rasX, rasY rasY and rasZ rasZ form a separate phylogenetic clade."""
129,131,0.980153,135,137,"""An analysis of Dictyostelium ras family genes reveals that rasU rasU, rasV rasV, rasW rasW, rasX rasX, rasY rasY and rasZ form a separate phylogenetic clade.""","""An analysis of Dictyostelium ras family genes reveals that rasU rasU, rasV rasV, rasW, rasX rasX, rasY rasY and rasZ rasZ form a separate phylogenetic clade."""
129,132,0.980153,135,138,"""An analysis of Dictyostelium ras family genes reveals that rasU rasU, rasV rasV, rasW rasW, rasX rasX, rasY rasY and rasZ form a separate phylogenetic clade.""","""An analysis of Dictyostelium ras family genes reveals that rasU rasU, rasV, rasW rasW, rasX rasX, rasY rasY and rasZ rasZ form a separate phylogenetic clade."""


I manually look at around the top 2000 results and a threshold of 
sim_th = 0.65
shall be fine

In [49]:
# -------------------------
# 1) CLUSTERING (union-find)
# -------------------------
sim_th = 0.6 #<------------I decided after maual inspection and iterate with later process in #4

parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# union candidate pairs above threshold
for i, j, sim in pairs:
    if sim >= sim_th:
        union(i, j)

# assign group ids (1..K)
roots = [find(i) for i in range(n)]
root_to_gid = {}
gids = []
for r in roots:
    if r not in root_to_gid:
        root_to_gid[r] = len(root_to_gid) + 1
    gids.append(root_to_gid[r])

claim_group_map = pl.DataFrame({
    "claim_id": ids,
    "group_claim_id": gids,
})

# quick cluster size stats
cluster_sizes = claim_group_map.group_by("group_claim_id").len().sort("len", descending=True)
display(cluster_sizes.head(30))

group_claim_id,len
i64,u32
147,29
16,25
457,16
445,13
62,9
…,…
54,3
623,3
882,3


In [51]:
# (4a_claim_groups.parquet is written after `rep` is defined — includes claim text + genes + canonical query.)

In [476]:
# ---------------------------------------------
# 2) TABLE: each group with variants in one row
# ---------------------------------------------
# Put variant claim_ids + variant texts into a list (for manual check)
group_variants = (
    claim_group_map
    .join(claims_u.select(["claim_id", "claim_plain", "claim_sim"]), on="claim_id", how="left")
    .group_by("group_claim_id")
    .agg([
        pl.col("claim_id").sort().alias("variant_claim_ids"),
        pl.col("claim_plain").alias("variant_claim_plain"),
        pl.col("claim_sim").alias("variant_claim_sim"),
        pl.len().alias("n_variants"),
    ])
    .sort(["n_variants", "group_claim_id"], descending=[True, False])
)

# Save for manual inspection
# TSV cannot store nested lists cleanly, so we stringify lists with join separators
group_variants_flat = (
    group_variants
    .with_columns([
        pl.col("variant_claim_ids")
          .list.eval(pl.element().cast(pl.Utf8)).list.join(",")
          .alias("variant_claim_ids"),
        pl.col("variant_claim_plain")
          .list.join(" ||| ")
          .alias("variant_claim_plain"),
        pl.col("variant_claim_sim")
          .list.join(" ||| ")
          .alias("variant_claim_sim"),
    ])
)

# group_variants_flat.write_csv("../tmp/group_variants.tsv", separator="\t")  # tmp/ removed
print("Wrote: tmp/group_variants.tsv")

Wrote: tmp/group_variants.tsv


In [477]:
# ------------------------------------------------------
# 3) Build a canonical representative text per group
#    (for downstream golden set + second TF-IDF peek)
# ------------------------------------------------------
# Choose canonical as the longest claim_plain (often most complete)
group_canon = (
    claim_group_map
    .join(claims_u.select(["claim_id", "claim_plain", "claim_sim"]), on="claim_id", how="left")
    .with_columns(pl.col("claim_plain").str.len_chars().alias("nchar"))
    .sort(["group_claim_id", "nchar"], descending=[False, True])
    .group_by("group_claim_id")
    .agg([
        pl.first("claim_plain").alias("canon_claim_plain"),
        pl.first("claim_sim").alias("canon_claim_sim"),
        pl.col("claim_id").sort().alias("variant_claim_ids"),
        pl.len().alias("n_variants"),
    ])
    .sort(["n_variants", "group_claim_id"], descending=[True, False])
)

# Save canonical table (flat)
group_canon_flat = group_canon.with_columns(
    pl.col("variant_claim_ids").list.eval(pl.element().cast(pl.Utf8)).list.join(",").alias("variant_claim_ids")
)
# group_canon_flat.write_csv("../tmp/group_canonical.tsv", separator="\t")  # tmp/ removed
print("Wrote: tmp/group_canonical.tsv")

Wrote: tmp/group_canonical.tsv


In [484]:
# ---------------------------------------------------------
# 4) PEEK #2: run TF-IDF again on canonical group texts
# ---------------------------------------------------------
group_texts = group_canon.get_column("canon_claim_sim").to_list()
group_ids   = group_canon.get_column("group_claim_id").to_list()
g = len(group_ids)

vec2 = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
X2 = vec2.fit_transform(group_texts)

k2 = 15
nn2 = NearestNeighbors(n_neighbors=min(k2, g), metric="cosine").fit(X2)
dist2, idx2 = nn2.kneighbors(X2, return_distance=True)

pairs2 = []
for i in range(g):
    for d, j in zip(dist2[i], idx2[i]):
        if j <= i:
            continue
        sim = 1.0 - float(d)
        pairs2.append((i, j, sim))
pairs2.sort(key=lambda x: x[2], reverse=True)

topN2 = 10
pairs2_df = pl.DataFrame({
    "i": [p[0] for p in pairs2[:topN2]],
    "j": [p[1] for p in pairs2[:topN2]],
    "sim": [p[2] for p in pairs2[:topN2]],
    "group_id_i": [group_ids[p[0]] for p in pairs2[:topN2]],
    "group_id_j": [group_ids[p[1]] for p in pairs2[:topN2]],
    "text_i": [group_texts[p[0]] for p in pairs2[:topN2]],
    "text_j": [group_texts[p[1]] for p in pairs2[:topN2]],
}).sort("sim", descending=True)

with pl.Config(fmt_str_lengths=500, tbl_rows=topN2, tbl_cols=20):
    display(pairs2_df)

# pairs2_df.write_csv("../tmp/group_pairwise_peek.tsv", separator="\t")  # tmp/ removed
print("Wrote: tmp/group_pairwise_peek.tsv")

i,j,sim,group_id_i,group_id_j,text_i,text_j
i64,i64,f64,i64,i64,str,str
389,390,0.597185,275,276,"""cotA encodes SP96 which is fucosylated before exocytosis from the PSVs and positioning in the outer layer of the spore coats.""","""cotB encodes SP70 which is glycosylated before being released and positioned in the outer layer of the spore coats."""
24,867,0.591388,83,813,"""aif downregulation also resulted in increased mitochondrial fission and defects in mitochondrial structure, further revealing the role of aif in mitochondrial organization and homeostasis.""","""Knockdown of DJ-1 also causes a small but significant activation of mitochondrial respiratory function, while overexpression inhibits mitochondrial activity, revealing that reduced levels of deeJ do not phenocopy mitochondrial dysfunction."""
1482,1483,0.590647,1477,1478,"""The X-ray structure of nucleoside diphosphate kinase NDP kinase from Dictyostelium discoideum has been determined.""","""The X-ray structure of wild type and several mutants nucleoside diphosphate kinase bound to substrates, co-factors and inhibitors has also been determined."""
113,608,0.584258,870,529,"""mRNA levels of racI increase during sexual development, suggesting that racI may function in sexual development.""","""gefB expression also increases during sexual development."""
364,1491,0.578504,249,1487,"""Characterization of cDNA and genomic clones of uppA showed that it encodes UDP-glucose pyrophosphorylase.""","""There is a second gene encoding UDP-glucose pyrophosphorylase, uppA uppA."""
638,639,0.577355,564,565,"""gpaA encodes G-alpha 1, one of the dozen alpha subunits that associate with the unique G-beta and G-gamma subunits gpbA, gpgA.""","""gpaB encodes one of the dozen G alpha subunits in Dictyostelium, Galpha2, which associates with Gbeta and Ggamma subunits gpbA, gpgA to form the G protein that binds to the cAMP receptor CAR1 carA-1, carA-2.."""
800,1142,0.577282,740,1117,"""In response to cAMP, RasG activation was greatly reduced in a gefR null mutant, indicating that GefR has GEF activity towards RasG.""","""RasG activation is reduced in gefR gefR and gflB gflB null mutant strains, indicating that these rasGEFs activate rasG.."""
170,1410,0.577016,20,1398,"""A histidine moeity in the 28 kDa RdeA protein accepts a phosphate from an aspartate in histidine kinases, and relays it to an aspartate in the response regulator region of the cAMP phosophodiesterase RegA regA.""","""The phosphate on RdeA is in equilibrium with phosphate on an aspartate in the response regulator region of the cAMP phosphodiesterase RegA regA."""
137,510,0.574659,1258,414,"""The adaptor protein ElmoE elmoE associates with the Rho GTPase RacB racB as well as with both ZizA and DocC docC, suggesting that ElmoE, DocC, and ZizA may form an evolutionarily conserved Elmo/Dock complex to serve as a GEF for RacB.""","""ElmoE elmoE associates with the Gbeta/Ggamma subunits and interacts with Dock-like proteins DocC docC and possibly ZizA zizA to activate the small GTPase RacB racB and promote actin polymerization in the leading edge of migrating cells."""


Wrote: tmp/group_pairwise_peek.tsv


In [485]:
claim_group_map.select(pl.col("group_claim_id").n_unique().alias("n_group_claim_id"))

n_group_claim_id
u32
1705


we still have 1705 claims. This is nice

let us get our golden set

In [523]:
# --- 1) attach group ids to the long evidence table ---
gold_long = (
    claim_cleaned_pmid_nonNA_abstract
    .join(claim_group_map, on="claim_id", how="left")
    .join(claims_u.select(["claim_id", "claim_sim"]), on="claim_id", how="left")
)

In [524]:
# --- 2) pick one canonical query per group (longest claim_sim) ---
# canon_query = (
#     gold_long
#     .select(["group_claim_id", "claim_id", "claim_sim"])
#     .unique(subset=["group_claim_id", "claim_id"])
#     .with_columns(pl.col("claim_sim").str.len_chars().alias("nchar"))
#     .sort(["group_claim_id", "nchar"], descending=[False, True])
#     .group_by("group_claim_id")
#     .agg(pl.first("claim_sim").alias("query"))
# )
rep = (
    gold_long
    .select(["group_claim_id", "claim_id", "claim_sim"])
    .unique(subset=["group_claim_id", "claim_id"])   # one per variant
    .with_columns(pl.col("claim_sim").str.len_chars().alias("nchar"))
    .sort(["group_claim_id", "nchar"], descending=[False, True])
    .group_by("group_claim_id")
    .agg([
        pl.first("claim_id").alias("rep_claim_id"),
        pl.first("claim_sim").alias("query"),
    ])
)
rep

group_claim_id,rep_claim_id,query
i64,i64,str
1,1,"""A basic region in the tail is …"
2,2,"""A cDNA clone derived from psvA…"
3,3,"""A cDNA clone of this gene, D19…"
4,4,"""A co-immunoprecipitation assay…"
5,5,"""A comparison of the genes tran…"
6,7,"""A DDB_G0269670 null mutant is …"
7,11,"""A Dictyostelium homolog of Vps…"
8,756,"""However, a double carA / carC …"
9,13,"""A double mutant in disgorgin a…"


In [ ]:
gene_by_claim = (
    claim_cleaned_pmid_nonNA_abstract
    .group_by("claim_id")
    .agg(
        pl.col("gene_id").unique().sort().str.join(",").alias("gene_id"),
    )
)
claim_groups_detail = (
    claim_group_map
    .join(claims_u, on="claim_id", how="left")
    .join(gene_by_claim, on="claim_id", how="left")
    .join(
        rep.select(["group_claim_id", "rep_claim_id", "query"]).rename({"query": "canonical_query"}),
        on="group_claim_id",
        how="left",
    )
    .with_columns(
        (pl.col("claim_id") == pl.col("rep_claim_id")).alias("is_representative_claim"),
    )
)
claim_groups_detail.write_parquet(OUTPUT_GOLD_BUILD / "4a_claim_groups.parquet")

In [525]:
# 3) Count variants (metadata) — from the mapping, not from citations
variant_counts = (
    claim_group_map
    .group_by("group_claim_id")
    .agg(pl.len().alias("n_variants"))
)

# 4) Keep ONLY the representative claim’s citations + anchors, then build golden set
rep_long = (
    gold_long
    .join(rep.select(["group_claim_id", "rep_claim_id"]), on="group_claim_id", how="inner")
    .filter(pl.col("claim_id") == pl.col("rep_claim_id"))
)

# 5) doc-level info (includes anchor_pos + citation_captions) for the representative query only
docs = (
    rep_long
    .group_by(["group_claim_id", "publication_id", "pmid", "title", "abstract_clean", "year"])
    .agg([
        pl.col("anchor_pos").drop_nulls().unique().sort().alias("anchor_pos"),
        pl.col("citation_captions").drop_nulls().unique().sort().alias("citation_captions"),
    ])
    .group_by("group_claim_id")
    .agg([
        pl.struct([
            "publication_id", "pmid", "title", "abstract_clean", "year",
            "anchor_pos", "citation_captions"
        ]).alias("docs"),
        pl.col("year").drop_nulls().unique().sort().alias("years"),   # years for THIS rep query only
    ])
)

gold = (
    rep
    .join(variant_counts, on="group_claim_id", how="left")
    .join(docs, on="group_claim_id", how="left")
    .with_columns([
        pl.col("docs").list.len().alias("n_citations"),
        pl.col("query").str.count_matches(r"\S+").alias("query_n_words"),
    ])
    .select([
        "group_claim_id",
        "rep_claim_id",
        "query",
        "n_variants",
        "n_citations",
        "query_n_words",
        "years",
        "docs",
    ])
)

gold.head(2)

group_claim_id,rep_claim_id,query,n_variants,n_citations,query_n_words,years,docs
i64,i64,str,u32,u32,u32,list[i32],list[struct[7]]
1,1,"""A basic region in the tail is …",1,1,17,[2014],"[{13954,""24747353"",""The association of myosin IB with actin waves in dictyostelium requires both the plasma membrane-binding site and actin-binding region in the myosin tail."",""F-actin structures and their distribution are important determinants of the dynamic shapes and functions of eukaryotic cells. Actin waves are F-actin formations that move along the ventral cell membrane driven by actin polymerization. Dictyostelium myosin IB is associated with actin waves but its role in the wave is unknown. Myosin IB is a monomeric, non-filamentous myosin with a globular head that binds to F-actin and has motor activity, and a non-helical tail comprising a basic region, a glycine-proline-glutamine-rich region and an SH3-domain. The basic region binds to acidic phospholipids in the plasma membrane through a short basic-hydrophobic site and the Gly-Pro-Gln region binds F-actin. In the current work we found that both the basic-hydrophobic site in the basic region and the Gly-Pro-Gln region of the tail are required for the association of myosin IB with actin waves. This is the first evidence that the Gly-Pro-Gln region is required for localization of myosin IB to a specific actin structure in situ. The head is not required for myosin IB association with actin waves but binding of the head to F-actin strengthens the association of myosin IB with waves and stabilizes waves. Neither the SH3-domain nor motor activity is required for association of myosin IB with actin waves. We conclude that myosin IB contributes to anchoring actin waves to the plasma membranes by binding of the basic-hydrophobic site to acidic phospholipids in the plasma membrane and binding of the Gly-Pro-Gln region to F-actin in the wave."",2014,[94],[""Brzeska et al. 2014""]}]"
2,2,"""A cDNA clone derived from psvA…",1,1,19,[1983],"[{8316,""6301681"",""Regulation of dictyostelium discoideum mRNAs specific for prespore or prestalk cells."",""Prespore and prestalk cells in Dictyostelium discoideum aggregates can be separated by density gradient centrifugation. Using poly(A+) RNA from the fractionated cells to probe a cDNA library of mRNAs from postaggregation cells, we were able to identify six cDNA clones representing RNAs enriched in prespore or prestalk cells. Remarkably, transcripts of six of seven cDNA clones, previously selected to encode mRNAs present in postaggregating cells but low or absent in growing cells, also are enriched in RNA from either prestalk or prespore cells. By hybridization of cDNA probes to nitrocellulose blots of formaldehyde RNA gels, these 13 mRNA species have been examined with respect to cell type specificity, temporal pattern of accumulation, and affect of disaggregation and cAMP on accumulation. Aggregation-stage mRNAs tend to fit into three different classes. All prespore mRNAs are similar in all aspects of their regulation, while prestalk mRNAs fall into two co-regulated classes. All mRNAs that are present at significant levels during growth and differentiation are found in both cell types at comparable levels. Our results indicate that there is coordinate control of expression of genes specific for the two principal cell types."",1983,[102],[""Barklis and Lodish 1983""]}]"


In [526]:
gold.write_parquet(OUTPUT_GOLD_BUILD / "4b_golden_grouped.parquet")

In [527]:
gold_flat = (
    gold
    .explode("docs")
    .unnest("docs")
    .select([
        "group_claim_id",
        "query",
        "n_variants",
        "n_citations",
        "publication_id",
        "pmid",
        "title",
        "abstract_clean",
        "year",
    ])
)

# gold_flat.write_csv("../tmp/golden_flat.tsv", separator="\t")  # tmp/ removed
print("Wrote: tmp/golden_flat.tsv")

Wrote: tmp/golden_flat.tsv


let us check out query statistics

In [528]:
# number of the citation a query have
cit_dist = gold.group_by("n_citations").len().sort("n_citations")
cit_dist

n_citations,len
u32,u32
1,1390
2,248
3,51
4,11
5,2
7,1
9,1
10,1


In [529]:
many_cit = gold.filter(pl.col("n_citations") > 5).select([
    "group_claim_id",
    "n_citations",
    "n_variants",
    "query",
])

with pl.Config(fmt_str_lengths=10_000, tbl_rows=500, tbl_cols=20):
    display(many_cit.sort("n_citations", descending=True))

group_claim_id,n_citations,n_variants,query
i64,u32,u32,str
511,10,1,"""G boxes are found in the promoters of several postaggregative and cell-type-specific genes, including carC, cotB, cotC, cprA, cprB, and ecmB. Consistent with these data, analysis of the gbfA -null cell line provided additional evidence that GBF is a key component of the development switch between aggregation and multicellular differentiation as a general inducer of postaggregative and cell-type-specific gene expression multicellular organismal development."""
183,9,1,"""Ca++/calmodulin binds to many Dictyostelium proteins including a ribosomal protein L19 rpl19, cyclin-dependent kinase cdk5 and nucleomorphin numA among others."""
1071,7,6,"""These include: the protein kinase PakA pakA, PakC pakC, Lissencephaly-related protein LIS1 lis1, diaphanous-related formin H forH, the myosin heavy chain kinase PakB pakB pakB, coronin A corA and WASP wasA."""


In [530]:
rep.filter(pl.col("group_claim_id") == 1071).select(["group_claim_id", "rep_claim_id", "query"])
gene = (
    claim_cleaned_pmid_nonNA_abstract
    .filter(pl.col("claim_id") == 1835)
    .select(pl.col("gene_id").unique())
)

gene

gene_id
str
"""DDB_G0277869"""


Looked at http://dictybase.org/gene/DDB_G0277869, it is fine.


In [531]:
len_dist = (
    gold.with_columns(
        pl.when(pl.col("query_n_words") < 10).then(pl.lit("<10"))
          .when(pl.col("query_n_words") < 20).then(pl.lit("10-19"))
          .when(pl.col("query_n_words") < 40).then(pl.lit("20-39"))
          .otherwise(pl.lit("40+"))
          .alias("len_bucket")
    )
    .group_by("len_bucket")
    .len()
    .sort("len_bucket")
)
len_dist

len_bucket,len
str,u32
"""10-19""",686
"""20-39""",872
"""40+""",77
"""<10""",70


In [532]:
var_dist = (
    gold.with_columns(
        pl.when(pl.col("n_variants") == 1).then(pl.lit("1"))
          .when(pl.col("n_variants") <= 3).then(pl.lit("2-3"))
          .when(pl.col("n_variants") <= 10).then(pl.lit("4-10"))
          .otherwise(pl.lit("11+"))
          .alias("variant_bucket")
    )
    .group_by("variant_bucket")
    .len()
    .sort("variant_bucket")
)

var_dist

variant_bucket,len
str,u32
"""1""",1549
"""11+""",4
"""2-3""",135
"""4-10""",17


In [535]:
# year
year_dist = (
    gold
    .select(["group_claim_id", "years"])
    .explode("years")
    .group_by("years")
    .len()
    .sort("years")
)
year_dist



years,len
i32,u32
null,3
1969,4
1970,2
1973,1
1976,8
1978,3
1979,1
1981,3
1982,1


conclusion of thses statistics:
- 1-citation queries is domindated.
- sentence length looks fine.
- there are some big  vairant groups

Thoughts:
- This could be a good automatic dataset
- If we should have a second track to manually pick a smaller gold subset
    - this need to be very accurate
    - in this case we shall keep oversample multi-citation groups, i.e those n_citations>1
- those big variant_bucket might be helpful for stress testing / hard negatives (they are similar but different citations), for now variant_bucket is merged